In [ ]:
#@title Cell 40.1 — Overview and validation boundary
from IPython.display import Markdown, display

display(Markdown("""
# NB40 — BMC: Training-only Model C calibration data

**Purpose.** Generate Model C predictions on **inner held-out pathogens drawn only from each outer training fold**, for future Bayesian Model C (BMC-N, BMC-T, BMC-AT) calibration. No Bayesian model is fitted in NB40.

**Inputs:** Three original ZIP archives from Notebooks 16B, 25 and 26. Their locations are found automatically anywhere under the mounted Google Drive; no manual paths, moving files, or extraction are necessary.

**Method:** For each original outer fold, retain the Model C kernel candidate, embedding dimension and Ridge penalty selected by Notebook 26 using only that outer fold's training data. Refit Model C in three inner pathogen-out splits and save inner out-of-fold predictions for the outer training pathogens. Keep the original independent outer held-out predictions for later BMC testing.

**Strict outer-test boundary:** BMC calibration for outer fold *k* must use *only* the inner predictions labelled `outer_fold_for_calibration=k`. A training pathogen's own MIC is excluded when generating its inner held-out Model C prediction. The outer-test MICs are never used in this notebook's fitting or hyperparameter selection. Outer-test MICs are read solely for a numerical reconstruction audit of Notebook 26 and exported unchanged for future evaluation.

**Important nuance:** The Model C settings were selected within each outer training fold by Notebook 26's inner-CV procedure; the selected settings are reused for inner out-of-fold calibration predictions. Thus the outer-test evaluation remains independent, while inner errors are conditional on training-fold hyperparameter selection. Genomic kernels and embeddings were constructed from cohort-wide genomic inputs, as in the original Model C (transductive feature construction).

**Execution:** Run Cells 40.1–40.7 in order. If an intensive computation is interrupted, rerun from Cell 40.5; validated completed folds and spectral embeddings are reused. Results are saved in `RefMIC/03_Results/40_BMC_Training_Only_Calibration/`.
"""))


In [ ]:
#@title Cell 40.2 — Mount Drive and automatically locate the three original archives
import os, json, gc, math, hashlib, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse.linalg import eigsh
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from IPython.display import display

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get('BMC_PROJECT_ROOT','/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC folder not found at {ROOT}. Set BMC_PROJECT_ROOT for a different location.')
SEARCH_ROOT = Path('/content/drive') if Path('/content/drive').is_dir() else ROOT
OUT = ROOT/'03_Results'/'40_BMC_Training_Only_Calibration'
OUT.mkdir(parents=True,exist_ok=True)
WORK = Path('/content/nb40_work') if Path('/content').is_dir() else OUT/'temporary_input_cache'
WORK.mkdir(parents=True,exist_ok=True)

ARCHIVE_NAMES = {
    'nb26': '26_model_c_nested_pathogen_out_and_reference_model_outputs.zip',
    'nb25': '25_model_c_pathogen_kernel_candidate_components.zip',
    'nb16b': '16B_optimised_full_gene_model3_interaction_outputs.zip',
}

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def discover_files(names,base=SEARCH_ROOT):
    """One drive walk for exact basenames; safely handle duplicate copies."""
    targets=set(names)
    found={n:[] for n in names}
    for folder, dirs, files in os.walk(base):
        dirs[:]=[d for d in dirs if d not in {'.Trash-1000','.Trashes','.git','node_modules','__pycache__'}]
        for fname in targets.intersection(files):
            found[fname].append(Path(folder)/fname)
    return found

archive_found=discover_files(ARCHIVE_NAMES.values())
ARCHIVES={}
for key,name in ARCHIVE_NAMES.items():
    hits=sorted(archive_found[name],key=lambda x:(0 if ROOT in x.parents else 1,len(x.parts),str(x)))
    if not hits:
        raise FileNotFoundError(f'Could not locate {name} anywhere under {SEARCH_ROOT}.')
    if len(hits)>1:
        hashes={sha256(p) for p in hits}
        if len(hashes)!=1:
            raise ValueError('Different copies of '+name+' found; specify correct original.\n'+'\n'.join(map(str,hits)))
        print(f'{name}: {len(hits)} identical copies found; using closest to RefMIC.')
    ARCHIVES[key]=hits[0]
    # Validate the ZIP central directory now; CRC is checked as each needed member is read.
    with zipfile.ZipFile(hits[0]) as archive:
        if not archive.namelist():
            raise ValueError(f'Empty ZIP archive: {hits[0]}')
    print(key+':',hits[0])
print('Results will be written to:',OUT)


In [ ]:
#@title Cell 40.3 — Read original Model C data and selected outer-fold settings
def extract_member(archive_key, member_basename):
    """Extract one named member; never extract the entire ZIP."""
    target=WORK/(archive_key+'__'+member_basename)
    with zipfile.ZipFile(ARCHIVES[archive_key]) as z:
        candidates=[n for n in z.namelist() if Path(n).name==member_basename and not n.endswith('/')]
        if len(candidates)!=1:
            raise FileNotFoundError(f'{member_basename}: expected once in {ARCHIVES[archive_key].name}, found {len(candidates)}')
        info=z.getinfo(candidates[0])
        if target.is_file() and target.stat().st_size==info.file_size:
            return target
        partial=Path(str(target)+'.partial')
        with z.open(info) as src,open(partial,'wb') as dest:
            shutil.copyfileobj(src,dest,4*1024*1024)
        partial.replace(target)
    return target

PRED_PATH=extract_member('nb26','26_nested_pathogen_out_predictions.csv.gz')
INTER_PATH=extract_member('nb26','26_model_c_aligned_interactions.csv.gz')
CATALOG_PATH=extract_member('nb26','26_model_c_candidate_embedding_catalog.csv')
ABX_PATH=extract_member('nb16b','16B_optimised_model3_kernel_embeddings.npz')
ORDER_PATH=extract_member('nb25','25_model_c_pathogen_order.csv')

inter=pd.read_csv(INTER_PATH,compression='gzip')
pred=pd.read_csv(PRED_PATH,compression='gzip')
pred=pred.loc[pred['model'].eq('Model C')].copy().sort_values('interaction_row').reset_index(drop=True)
catalog=pd.read_csv(CATALOG_PATH)
order=pd.read_csv(ORDER_PATH).sort_values('model_c_row_index').reset_index(drop=True)
with np.load(ABX_PATH,allow_pickle=False) as z:
    abx=np.asarray(z['antibiotic_embedding'],dtype=np.float32)

if inter.shape[0]!=50460 or pred.shape[0]!=50460 or inter.biosample.nunique()!=9058 or inter.antibiotic.nunique()!=26:
    raise ValueError('Unexpected Model C observation or pathogen counts.')
if abx.shape!=(26,26) or len(order)!=9058:
    raise ValueError('Unexpected antibiotic or pathogen coordinate dimensions.')
if not np.array_equal(order['model_c_row_index'].to_numpy(),np.arange(9058)):
    raise ValueError('Notebook 25 pathogen row order is inconsistent.')
if not np.array_equal(inter['biosample'].astype(str).to_numpy(), order.biosample.astype(str).to_numpy()[inter.model_c_pathogen_row.to_numpy(dtype=int)]):
    raise ValueError('Model C observation pathogen rows do not match Notebook 25 order.')
if not np.array_equal(pred.interaction_row.to_numpy(dtype=int),np.arange(50460)):
    raise ValueError('Outer prediction identity/order is incomplete.')
for c1,c2 in [('biosample','biosample'),('antibiotic','antibiotic')]:
    if not np.array_equal(inter[c1].astype(str),pred[c2].astype(str)):
        raise ValueError(f'Original MIC and prediction identity mismatch: {c1}.')
if not np.allclose(inter.log2_mic,pred.observed_log2_mic,rtol=0,atol=1e-10):
    raise ValueError('Original and held-out observed MIC values disagree.')
if pred.duplicated(['biosample','antibiotic']).any():
    raise ValueError('Duplicate observations in Model C held-out predictions.')
if (pred.groupby('biosample').outer_fold.nunique()!=1).any():
    raise ValueError('Outer folds are not pathogen-disjoint.')
if set(pred.outer_fold.unique())!={1,2,3,4,5}:
    raise ValueError('Expected exactly 5 outer folds.')

fold_settings=pred.groupby('outer_fold',sort=True)[['candidate_id','rho','pathogen_coordinates','alpha']].first().reset_index()
for column in ['candidate_id','rho','pathogen_coordinates','alpha']:
    if (pred.groupby('outer_fold')[column].nunique()!=1).any():
        raise ValueError(f'Outer-fold settings vary within a fold: {column}')
if not set(fold_settings.candidate_id).issubset(set(catalog.candidate_id)):
    raise ValueError('Selected candidate absent from Notebook 26 catalog.')

pathrow=inter.model_c_pathogen_row.to_numpy(dtype=np.int32)
antirow=inter.antibiotic_coordinate_row.to_numpy(dtype=np.int32)
y=inter.log2_mic.to_numpy(dtype=np.float64)
groups=inter.biosample.astype(str).to_numpy()
outer=pred.outer_fold.to_numpy(dtype=np.int8)
assert np.isfinite(y).all() and np.isfinite(pred.predicted_log2_mic).all()
assert np.all((pathrow>=0)&(pathrow<9058)) and np.all((antirow>=0)&(antirow<26))

print('Matched: 50,460 observations; 9,058 BioSamples; 26 antibiotics; five outer folds.')
display(fold_settings)


In [ ]:
#@title Cell 40.4 — Find or reconstruct the two original selected Model C embeddings
# Use the exact Notebook 26 embeddings when they exist on the current Drive.
# Otherwise reconstruct them from the two Notebook 25 kernel components.
# Reconstruction can be computationally expensive; results are checkpointed.

EMB_DIR=OUT/'embeddings'
SPEC_DIR=OUT/'spectra'
EMB_DIR.mkdir(parents=True,exist_ok=True)
SPEC_DIR.mkdir(parents=True,exist_ok=True)
needed_ids=sorted(fold_settings.candidate_id.unique())
original_embed_names=[f'26_{cid}_embedding.npz' for cid in needed_ids]
found_embed=discover_files(original_embed_names)
original_catalog=catalog.set_index('candidate_id')

# Sign normalization, exactly as Notebook 26.
def normalise_signs(evec):
    v=np.asarray(evec,dtype=np.float64).copy()
    for j in range(v.shape[1]):
        pivot=np.argmax(np.abs(v[:,j]))
        if v[pivot,j]<0: v[:,j]*=-1
    return v

EMBEDDINGS={}
missing=[]
for cid in needed_ids:
    fname=f'26_{cid}_embedding.npz'
    # Reuse Notebook 26 checkpoint only if its SHA matches its original catalog.
    paths=sorted(found_embed.get(fname,[]),key=lambda p:(0 if ROOT in p.parents else 1,str(p)))
    paths+=([EMB_DIR/fname] if (EMB_DIR/fname).is_file() else [])
    expected_hash=str(original_catalog.loc[cid,'checkpoint_sha256'])
    chosen=None
    for p in paths:
        if p.exists() and sha256(p)==expected_hash:
            chosen=p;break
    if chosen is not None:
        with np.load(chosen,allow_pickle=False) as z:
            a=np.asarray(z['pathogen_embedding'],dtype=np.float32)
        if a.shape!=(9058,256):
            raise ValueError(f'Invalid embedding shape for {cid}: {a.shape}')
        EMBEDDINGS[cid]=a
        print('Reusing exact Notebook 26 checkpoint:',chosen)
    else:
        # A notebook-40-generated embedding can be reused only after the
        # outer-prediction reconstruction audit has passed for every fold.
        p=EMB_DIR/fname
        marker=p.with_suffix('.verified.json')
        if p.is_file() and marker.is_file():
            with open(marker) as f: meta=json.load(f)
            if meta.get('sha256')==sha256(p) and meta.get('validation_status')=='passed':
                with np.load(p,allow_pickle=False) as z:
                    a=np.asarray(z['pathogen_embedding'],dtype=np.float32)
                if a.shape==(9058,256):
                    EMBEDDINGS[cid]=a
                    print('Reusing audited NB40 reconstructed embedding:',p)
                    continue
        missing.append(cid)

if missing:
    print('Notebook 26 selected embeddings not found. Reconstructing:',missing)
    # This fallback uses the same leading-512 component eigenspaces and
    # low-rank Gram combination as original Notebook 26.
    comp={}
    for kind,filename in [('model3b','25_model3b_pathogen_kernel_subset.npy'),
                          ('locus','25_sequence_kernel_locus_weighted.npy')]:
        spec_file=SPEC_DIR/f'40_{kind}_rank_512.npz'
        if spec_file.exists():
            with np.load(spec_file,allow_pickle=False) as z:
                eigvals=np.asarray(z['eigenvalues'],dtype=np.float64)
                eigvecs=np.asarray(z['eigenvectors'],dtype=np.float64)
        else:
            print('Computing component spectrum:',kind,'(this may be lengthy)')
            src=extract_member('nb25',filename)
            k=np.load(src,mmap_mode='r')
            if k.shape!=(9058,9058):raise ValueError(f'Unexpected {kind} kernel shape {k.shape}')
            v0=np.random.default_rng(42).normal(size=9058)
            w,v=eigsh(k,k=512,which='LA',v0=v0,tol=1e-4,maxiter=5000)
            sort=np.argsort(w)[::-1]
            eigvals=np.clip(np.asarray(w[sort],dtype=np.float64),0,None)
            eigvecs=normalise_signs(v[:,sort])
            np.savez_compressed(spec_file,eigenvalues=eigvals.astype(np.float32),
                                eigenvectors=eigvecs.astype(np.float32))
            del k,v,w
            gc.collect()
        if eigvals.shape!=(512,) or eigvecs.shape!=(9058,512):
            raise ValueError('Invalid '+kind+' spectrum')
        comp[kind]=eigvecs*np.sqrt(eigvals)[None,:]

    for cid in missing:
        c=original_catalog.loc[cid]
        rho=float(c['rho'])
        if c['sequence_component_id']!='locus' or not (0<rho<1):
            raise ValueError('Unexpected original selected candidate: '+cid)
        F=np.concatenate([np.sqrt(1-rho)*comp['model3b'],np.sqrt(rho)*comp['locus']],axis=1)
        gram=F.T@F
        eigval,eigvec=np.linalg.eigh(gram)
        indices=np.argsort(eigval)[::-1][:256]
        embedding=normalise_signs(F@eigvec[:,indices]).astype(np.float32)
        p=EMB_DIR/f'26_{cid}_embedding.npz'
        np.savez_compressed(p,pathogen_embedding=embedding,
                            pathogen_eigenvalues=np.clip(eigval[indices],0,None).astype(np.float32))
        EMBEDDINGS[cid]=embedding
        del F,gram,eigval,eigvec
        gc.collect()
        print('Reconstructed:',cid,'(will verify against original outer predictions)')
    del comp
    gc.collect()
else:
    print('All selected embeddings located; no spectral recomputation needed.')

assert set(EMBEDDINGS)==set(needed_ids)


In [ ]:
#@title Cell 40.5 — Define checked Model C prediction and restartable folds
# Mirrors the original Model C tensor-product interaction design and Ridge.
# NOTE: No BMC fit or predictive distribution is calculated here.

def design(indices,embedding,dimension):
    i=np.asarray(indices,dtype=np.int64)
    x=embedding[pathrow[i],:dimension].astype(np.float32,copy=False)
    a=abx[antirow[i]].astype(np.float32,copy=False)
    return np.einsum('nr,ns->nrs',x,a,optimize=True).reshape(len(i),dimension*26).astype(np.float32,copy=False)

def fit_predict(train_idx,test_idx,embedding,dimension,alpha):
    X=design(train_idx,embedding,dimension)
    Y=design(test_idx,embedding,dimension)
    model=Ridge(alpha=float(alpha),fit_intercept=True,solver='lsqr',tol=1e-4,
                max_iter=2000,copy_X=False)
    model.fit(X,y[train_idx])
    p=np.asarray(model.predict(Y),dtype=np.float64)
    del model,X,Y
    gc.collect()
    return p

def fold_file(fold):return OUT/f'40_training_only_inner_oof_outer_fold_{fold:02d}.csv.gz'
def report_file(fold):return OUT/f'40_outer_reconstruction_audit_{fold:02d}.json'

def fold_completed(fold):
    f=fold_file(fold); r=report_file(fold)
    if not(f.is_file() and r.is_file()):return False
    try:
        report=json.loads(r.read_text())
        if report.get('validation_status')!='passed' or report.get('csv_sha256')!=sha256(f):return False
        tab=pd.read_csv(f)
        train_ix=np.flatnonzero(outer!=fold)
        return (len(tab)==len(train_ix) and tab.interaction_row.is_unique
                and set(tab.interaction_row)==set(train_ix.tolist())
                and tab.outer_fold_for_calibration.eq(fold).all()
                and (tab.groupby('biosample').inner_fold.nunique()==1).all()
                and np.isfinite(tab.inner_oof_predicted_log2_mic).all())
    except Exception:
        return False

print('Completed folds already present:',[k for k in range(1,6) if fold_completed(k)])
print('Folds to generate:',[k for k in range(1,6) if not fold_completed(k)])


In [ ]:
#@title Cell 40.6 — Produce training-only inner cross-fitted predictions for five outer folds
# Checkpoint per complete outer fold. A disconnected Colab runtime can resume
# from this cell after earlier cells have been rerun; completed folds are reused.
# For a shorter session set MAX_NEW_FOLDS_PER_RUN=1 and rerun 40.6 until complete.
MAX_NEW_FOLDS_PER_RUN=5
run_folds=[f for f in range(1,6) if not fold_completed(f)][:MAX_NEW_FOLDS_PER_RUN]
print('Outer folds being processed now:',run_folds)

for fold in run_folds:
    row=fold_settings.loc[fold_settings.outer_fold.eq(fold)].iloc[0]
    cid=row.candidate_id; emb=EMBEDDINGS[cid]
    dim=int(row.pathogen_coordinates); alpha=float(row.alpha)
    train_ix=np.flatnonzero(outer!=fold)
    test_ix=np.flatnonzero(outer==fold)
    assert set(groups[train_ix]).isdisjoint(set(groups[test_ix]))
    
    # Reproduce Notebook 26's original outer held-out predictions using
    # the same training records and candidate coordinates. Test MIC values
    # are *not* used in fitting; this is an output-identical audit only.
    print(f'Fold {fold}: checking Model C reconstruction...')
    reproduced=fit_predict(train_ix,test_ix,emb,dim,alpha)
    original=pred.predicted_log2_mic.to_numpy(dtype=np.float64)[test_ix]
    diff=reproduced-original
    validation_rmse=float(np.sqrt(np.mean(diff**2)))
    validation_max=float(np.max(np.abs(diff)))
    print(f'  Original Model C reproduction: RMSE={validation_rmse:.6f}, max absolute difference={validation_max:.6f} log2 units')
    if validation_rmse>0.02 or validation_max>0.15:
        raise ValueError(
            f'Fold {fold}: reconstructed Model C predictions do not reproduce Notebook 26. '
            'Check whether Notebook 26 candidate embedding checkpoints are still available. '
            'Calibration output was NOT saved for this fold.')
    candidate_cache=EMB_DIR/f'26_{cid}_embedding.npz'
    if candidate_cache.is_file():
        verified_marker=candidate_cache.with_suffix('.verified.json')
        verified_marker.write_text(json.dumps({'candidate_id':cid,'sha256':sha256(candidate_cache),
                                              'validation_status':'passed',
                                              'model_c_reproduction_rmse_log2':validation_rmse},indent=2))
    del reproduced,original,diff
    gc.collect()

    # Exactly three pathogen-disjoint inner folds within original outer TRAIN.
    splitter=GroupKFold(n_splits=3)
    splits=list(splitter.split(np.zeros(len(train_ix),dtype=np.uint8),
                              y[train_ix],groups=groups[train_ix]))
    fitted=np.full(len(train_ix),np.nan,dtype=np.float64)
    inner_label=np.zeros(len(train_ix),dtype=np.int8)
    for inner,(rel_train,rel_eval) in enumerate(splits,start=1):
        actual_train=train_ix[rel_train];actual_eval=train_ix[rel_eval]
        if not set(groups[actual_train]).isdisjoint(set(groups[actual_eval])):
            raise AssertionError('Inner pathogen leakage detected')
        if not set(groups[actual_train]).isdisjoint(set(groups[test_ix])):
            raise AssertionError('Outer pathogen leakage detected')
        print(f'  Inner {inner}/3: train {len(actual_train):,}, predict {len(actual_eval):,} observations')
        fitted[rel_eval]=fit_predict(actual_train,actual_eval,emb,dim,alpha)
        inner_label[rel_eval]=inner
    if not np.isfinite(fitted).all() or (inner_label==0).any():
        raise AssertionError(f'Fold {fold}: training-only predictions incomplete')

    df=pd.DataFrame({
        'outer_fold_for_calibration':fold,
        'inner_fold':inner_label,
        'interaction_row':train_ix,
        'biosample':groups[train_ix],
        'antibiotic':inter.antibiotic.to_numpy()[train_ix],
        'observed_log2_mic':y[train_ix],
        'inner_oof_predicted_log2_mic':fitted,
        'training_residual_log2':y[train_ix]-fitted,
        'candidate_id':cid,
        'pathogen_coordinates':dim,
        'alpha':alpha,
    }).sort_values('interaction_row').reset_index(drop=True)
    if df.duplicated(['interaction_row']).any() or (df.groupby('biosample').inner_fold.nunique()!=1).any():
        raise AssertionError('Cross-fitting identity audit failed')
    f=fold_file(fold)
    part=Path(str(f)+'.partial')
    df.to_csv(part,index=False,compression='gzip')
    part.replace(f)
    report={
        'outer_fold':fold,'candidate_id':cid,'pathogen_coordinates':dim,'alpha':alpha,
        'n_calibration_rows':int(len(df)),'n_calibration_pathogens':int(df.biosample.nunique()),
        'n_outer_test_rows':int(len(test_ix)),'n_outer_test_pathogens':int(len(set(groups[test_ix]))),
        'model_c_reproduction_rmse_log2':validation_rmse,
        'model_c_reproduction_max_abs_log2':validation_max,
        'csv_sha256':sha256(f),'validation_status':'passed',
        'note':'Inner predictions generated without fitting to each inner-heldout pathogen; outer test excluded.'
    }
    report_file(fold).write_text(json.dumps(report,indent=2))
    print(f'Fold {fold}: saved {len(df):,} training-only predictions to {f.name}')
    gc.collect()

print('Completed:',[k for k in range(1,6) if fold_completed(k)])
print('Remaining:',[k for k in range(1,6) if not fold_completed(k)])


In [ ]:
#@title Cell 40.7 — Audit all five folds and export BMC calibration-ready datasets
incomplete=[k for k in range(1,6) if not fold_completed(k)]
if incomplete:
    print(f'Outer folds incomplete: {incomplete}. Rerun Cell 40.6; then rerun this cell.')
else:
    training=pd.concat([pd.read_csv(fold_file(k)) for k in range(1,6)],ignore_index=True)
    if len(training)!=4*50460:
        raise AssertionError('Expected 201,840 calibration interaction rows across folds')
    for k in range(1,6):
        t=training.loc[training.outer_fold_for_calibration.eq(k)]
        expected=set(np.flatnonzero(outer!=k).tolist())
        if set(t.interaction_row)!=expected:raise AssertionError(f'Incorrect calibration data in fold {k}')
        if set(t.biosample)&set(pred.loc[pred.outer_fold.eq(k),'biosample']):
            raise AssertionError(f'Outer test pathogens present in fold {k} calibration data')

    outpath=OUT/'40_BMC_training_only_inner_predictions_all_outer_folds.csv.gz'
    training.to_csv(outpath,index=False,compression='gzip')
    outer_test=(pred[['interaction_row','biosample','antibiotic','observed_log2_mic',
                      'predicted_log2_mic','outer_fold']]
                .rename(columns={'predicted_log2_mic':'model_c_outer_test_predicted_log2_mic'})
                .sort_values(['outer_fold','interaction_row']).reset_index(drop=True))
    testpath=OUT/'40_BMC_independent_outer_test_predictions.csv.gz'
    outer_test.to_csv(testpath,index=False,compression='gzip')
    details=pd.DataFrame([json.loads(report_file(k).read_text()) for k in range(1,6)])
    display(details[['outer_fold','candidate_id','n_calibration_rows','n_calibration_pathogens',
                     'model_c_reproduction_rmse_log2','model_c_reproduction_max_abs_log2']])
    details.to_csv(OUT/'40_BMC_fold_reconstruction_and_leakage_audit.csv',index=False)
    print('All five pathogen-out folds passed.')
    print('Calibration records (five training sets):',f'{len(training):,}')
    print('Independent Model C outer test records:',f'{len(outer_test):,}')
    print('Next: fit BMC independently per antibiotic and outer fold, using training-only inner predictions.')
    print('Saved:',outpath)
    print('Saved:',testpath)
